# Notebook 01: Exploratory Data Analysis (EDA) & Class Imbalance Analysis

- **Objective**: Objective 1 — Baseline ML Models & Data Understanding
- **Purpose**: Load the Cardiotocography (CTG) fetal health dataset, inspect data integrity and null values, plot target class distribution, and document class imbalance.
- **Outputs**: Target distribution summary, class imbalance plot, and numerical breakdown.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_data, TARGET_COLUMN

# Matplotlib visual configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('muted')


In [ ]:
# Load dataset via src/data/load.py
df = load_data(PROJECT_ROOT / "data/raw/fetal_health.csv")
print(f"Dataset Shape: {df.shape}")
print(f"Total Features: {df.shape[1] - 1} numeric features + 1 target ('{TARGET_COLUMN}')")


In [ ]:
# Check null values across all columns
null_summary = df.isnull().sum()
total_nulls = null_summary.sum()

print(f"Total Missing Values in Dataset: {total_nulls}")
print("\nFirst 5 rows of the dataset:")
display(df.head())


In [ ]:
# Target class mappings
class_names = {1.0: 'Normal (1)', 2.0: 'Suspect (2)', 3.0: 'Pathological (3)'}

# Calculate counts and percentages
counts = df[TARGET_COLUMN].value_counts().sort_index()
percentages = df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100

summary_df = pd.DataFrame({
    'Class Name': [class_names[c] for c in counts.index],
    'Count': counts.values,
    'Percentage (%)': percentages.values.round(2)
})

print("=== Target Class Numerical Distribution ===")
display(summary_df)


In [ ]:
# Plot Class Distribution Bar Chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

bars = ax.bar(summary_df['Class Name'], summary_df['Count'], color=colors, edgecolor='black', alpha=0.85)

# Annotate bars with exact count and percentage
for bar, count, pct in zip(bars, summary_df['Count'], summary_df['Percentage (%)']):
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.0,
        yval + 25,
        f"{count}\n({pct}%)",
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

ax.set_title("CTG Fetal Health Class Distribution", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Fetal Health Status", fontsize=12, labelpad=10)
ax.set_ylabel("Number of Patients (Count)", fontsize=12, labelpad=10)
ax.set_ylim(0, max(summary_df['Count']) * 1.15)
plt.tight_layout()
plt.show()


## Key EDA Findings & Imbalance Summary

1. **Clean Dataset**: The dataset contains 2,126 rows and 22 columns with **0 missing (null) values**.
2. **Severe Class Imbalance**:
   - **Normal (1.0)**: 1,655 cases (**77.85%**) — dominates the dataset.
   - **Suspect (2.0)**: 295 cases (**13.88%**) — moderate minority.
   - **Pathological (3.0)**: 176 cases (**8.28%**) — severe minority class.
3. **Implication for Modeling**:
   - Standard accuracy is a misleading metric here (a trivial model predicting "Normal" every time gets ~78% accuracy while failing 100% of high-risk cases).
   - Cost-sensitive weighting (`class_weight='balanced'`) or uncertainty calibration is essential to protect recall on the minority **Pathological** class.
